In [0]:
from pyspark.sql import functions as F

# ============================================================
# 1. CONFIGURATION
# ============================================================

storage_account = "stsupplychain2026xx"

gold_delivery_performance_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "gold/delivery_performance/"
)

gold_daily_shipments_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "gold/daily_shipments/"
)


# ============================================================
# 2. LECTURE
# ============================================================

delivery_performance = (
    spark.read
    .format("delta")
    .load(gold_delivery_performance_path)
)


# ============================================================
# 3. AGRÉGATION JOURNALIÈRE
# ============================================================

gold_daily_shipments = (
    delivery_performance

    .groupBy("shipment_date")

    .agg(

        # Nombre total de shipments créés ce jour-là
        F.count("*")
        .alias("total_shipments"),

        # Nombre de shipments déjà livrés
        F.sum(
            F.col("is_delivered").cast("int")
        )
        .alias("delivered_shipments"),

        # Nombre de shipments en retard
        F.sum(
            F.when(
                F.col("is_late") == True,
                1
            ).otherwise(0)
        )
        .alias("late_shipments"),

        # Nombre livrés à l'heure
        F.sum(
            F.when(
                (F.col("is_delivered") == True)
                & (F.col("is_late") == False),
                1
            ).otherwise(0)
        )
        .alias("on_time_shipments"),

        # Durée moyenne de livraison
        F.avg(
            "delivery_duration_hours"
        )
        .alias("avg_delivery_duration_hours")
    )
)


# ============================================================
# 4. TAUX DE LIVRAISON À L'HEURE
# ============================================================

gold_daily_shipments = (
    gold_daily_shipments

    .withColumn(
        "on_time_delivery_rate_pct",

        F.when(
            F.col("delivered_shipments") > 0,

            F.round(
                (
                    F.col("on_time_shipments")
                    / F.col("delivered_shipments")
                ) * 100,
                2
            )
        )

        .otherwise(
            F.lit(None)
        )
    )
)


# ============================================================
# 5. ÉCRITURE GOLD
# ============================================================

(
    gold_daily_shipments
    .write
    .format("delta")
    .mode("overwrite")
    .save(gold_daily_shipments_path)
)

print(
    "gold_daily_shipments créée avec succès."
)


# ============================================================
# 6. AFFICHAGE
# ============================================================

display(
    gold_daily_shipments
    .orderBy("shipment_date")
)